In [1]:
import pandas as pd
import sqlite3
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, recall_score
from joblib import dump, load
import openmeteo_requests
import tqdm
from statsmodels.tsa.arima.model import ARIMA
import warnings
import numpy as np

In [3]:
try:
    con = sqlite3.connect(r"C:\Users\demoexam\Desktop\Гурбанов\MachineLearning\Проект №6 (последний срез)\clear_data.db")
    data = pd.read_sql("SELECT * FROM Clear_data", con=con)
except Exception as e:
    print(f"Ошибка в загрузке из базы данных: {e}")

In [4]:
data = data.drop(["level_0", "index"], axis=1)

In [5]:
data = data.drop(["country", "cluster_2", "track_id", "point_id"], axis=1)

In [6]:
data

,time,latitude,longitude,elevation,steps,temperature,water_feature,forest_feature,buildings_feature,place_type,cluster,fire,flood,evacuation,dangerous
0,2012-09-12 21:13:13+00:00,51.505713,104.139225,501.46,3.0,-25.937000,1,3,94,2,2,1,0,0,2
1,2012-09-12 21:13:19+00:00,51.505694,104.139221,505.79,0.0,-25.937000,1,3,94,2,2,1,0,0,2
2,2012-09-12 21:13:20+00:00,51.505693,104.139224,505.79,5.0,-25.937000,1,3,94,2,2,1,0,0,2
3,2012-09-12 21:13:30+00:00,51.505724,104.139204,511.56,1.0,-25.937000,1,3,94,2,2,1,0,0,2
4,2012-09-12 21:13:37+00:00,51.505717,104.139202,514.92,1.0,-25.937000,1,3,94,2,2,1,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2741,2022-07-16 09:35:57+00:00,51.737582,103.428106,669.33,44.0,-29.246500,0,0,0,0,1,0,0,1,1
2742,2022-07-16 09:36:02+00:00,51.737811,103.427860,668.52,44.0,-29.246500,0,0,0,0,1,0,0,1,1
2743,2022-07-16 09:36:07+00:00,51.738017,103.427555,668.27,16.0,-29.246500,0,0,0,0,1,0,0,1,1
2744,2022-07-16 09:36:24+00:00,51.738100,103.427456,668.27,31.0,-29.207499,0,0,0,0,1,0,0,1,1


In [7]:
data["time"] = pd.to_datetime(data["time"])

In [8]:
data['date_day'] = data['time'].dt.day
data['date_month'] = data['time'].dt.month
data['date_year'] = data['time'].dt.year
data['date_hour'] = data['time'].dt.hour

In [9]:
data = data.drop("time", axis=1)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(data.drop("fire", axis=1), data["fire"], test_size=0.33, random_state=42)

In [20]:
LR = LogisticRegression(max_iter=5000)

In [21]:
LR.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,5000
,multi_class,'deprecated'


In [22]:
GBC = GradientBoostingClassifier()

In [23]:
GBC.fit(X_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [24]:
MLPC = MLPClassifier()

In [25]:
MLPC.fit(X_train, y_train)

,hidden_layer_sizes,"(100,)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,200
,shuffle,True
,random_state,None


In [26]:
models = {"MLPClassifier": MLPC, "GradientBoostingClassifier": GBC, "LogisticRegression": LR}

In [27]:
for name_model in models:
    model = models[name_model]
    model_pred = model.predict(X_test)
    model_proba = model.predict_proba(X_test)
    print(f"Модель {name_model}:\n\nТочность: {accuracy_score(y_test, model_pred)}\nПолнота: {recall_score(y_test, model_pred, average="micro")}\nF1-метрика: {f1_score(y_test, model_pred, average="micro")}\nROC-AUC метрика: {roc_auc_score(y_test, model_proba, average="micro", multi_class="ovr")}\n\n")

ValueError: y should be a 1d array, got an array of shape (907, 2) instead.

### Лучше всего будет выбрать алгоритм Логистической Регресии, так как она использует линейные зависимости, которые присуствуют в датасете. Помимо этого это достаточно простой алгоритм и он полностью справляется со своей задачей, брать более сложные алгоритмы смысла не имеет.

In [19]:
dump(models["LogisticRegression"], 'best_model.joblib')

['best_model.joblib']

In [20]:
data

,latitude,longitude,elevation,steps,temperature,water_feature,forest_feature,buildings_feature,place_type,cluster,fire,flood,evacuation,dangerous,date_day,date_month,date_year,date_hour
0,51.505713,104.139225,501.46,3.0,-25.937000,1,3,94,2,2,1,0,0,2,12,9,2012,21
1,51.505694,104.139221,505.79,0.0,-25.937000,1,3,94,2,2,1,0,0,2,12,9,2012,21
2,51.505693,104.139224,505.79,5.0,-25.937000,1,3,94,2,2,1,0,0,2,12,9,2012,21
3,51.505724,104.139204,511.56,1.0,-25.937000,1,3,94,2,2,1,0,0,2,12,9,2012,21
4,51.505717,104.139202,514.92,1.0,-25.937000,1,3,94,2,2,1,0,0,2,12,9,2012,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2741,51.737582,103.428106,669.33,44.0,-29.246500,0,0,0,0,1,0,0,1,1,16,7,2022,9
2742,51.737811,103.427860,668.52,44.0,-29.246500,0,0,0,0,1,0,0,1,1,16,7,2022,9
2743,51.738017,103.427555,668.27,16.0,-29.246500,0,0,0,0,1,0,0,1,1,16,7,2022,9
2744,51.738100,103.427456,668.27,31.0,-29.207499,0,0,0,0,1,0,0,1,1,16,7,2022,9


In [21]:
def __temperature_forecast(row):
    params = {
    "latitude": row["latitude"],
    "longitude": row["longitude"],
    "start_date": str(pd.to_datetime(f"{int(row["date_year"])}-{int(row["date_month"])}-{int(row["date_day"])}").strftime("%Y-%m-%d")),
    "end_date": str(pd.to_datetime(f"{int(row["date_year"])}-{int(row["date_month"])}-{int(row["date_day"])}").strftime("%Y-%m-%d")),
    "hourly": ["temperature_2m"]
    }
    try:
        openmeteo = openmeteo_requests.Client()
        responses = openmeteo.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)
        response = responses[0]
        current = response.Hourly()
        temperatures = current.Variables(0).ValuesAsNumpy()
        return float(temperatures.mean())
    except Exception as e:
        print(f"Ошибка получения температуры на точке: {(row["latitude"], row["longitude"])}\n{e}")
        return None
    

In [31]:
points_predictions_flood = {}
points_predictions_fire = {}



for i in tqdm.tqdm(range(len(data.index.tolist()))):
    time_column = []
    temperature_column = []
    temperature_range = pd.DataFrame()
    predictions = []
    row = data.iloc[i]
    for k in range(4):
        row["date_year"] -= 1
        time_column.append(str(pd.to_datetime(f"{int(row["date_year"])}-{int(row["date_month"])}-{int(row["date_day"])}").strftime("%Y-%m-%d")))
        temperature_column.append(__temperature_forecast(row))
    temperature_range = pd.concat([temperature_range, pd.Series(temperature_column, name="temperature")], axis=1)
    temperature_range = pd.concat([temperature_range, pd.Series(time_column, name="time")], axis=1)
    temperature_range["time"] = pd.to_datetime(temperature_range["time"])
    temperature_range = temperature_range.set_index("time")
    temperature_range = temperature_range.iloc[::-1]
    model = ARIMA(temperature_range, order=(1,1,1))
    model_fit = model.fit()
    arima_predictions = model_fit.forecast(steps=10)
    
    for k, v in enumerate(arima_predictions):
        row = data.iloc[i]
        row["date_year"] += 1
        row["temperature"] = v
        model = load(r"C:\Users\demoexam\Desktop\Гурбанов\MachineLearning\Проект №6 (последний срез)\best_model.joblib")
        predict = model.predict(pd.DataFrame(row).T.drop("dangerous", axis=1))
        predictions.append(int(predict))
    points_predictions_flood[f"point_{i}"] = predictions
    print(points_predictions_flood)
    
            

  0%|          | 0/2746 [00:00<?, ?it/s]C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 365D will be used.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 365D will be used.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 365D will be used.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few ob

{'point_0': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]}


C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 365D will be used.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 365D will be used.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 365D will be used.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting paramete

{'point_0': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_1': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]}


C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 365D will be used.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 365D will be used.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 365D will be used.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting paramete

...

{'point_0': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_1': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_2': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_3': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_4': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_5': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_6': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_7': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_8': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_9': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_10': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_11': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_12': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_13': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_14': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_15': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_16': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_17': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_18': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_19': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_20': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_21': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_22': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1

C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 365D will be used.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 365D will be used.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 365D will be used.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting paramete

{'point_0': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_1': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_2': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_3': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_4': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_5': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_6': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_7': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_8': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_9': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_10': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_11': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'point_12': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_13': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_14': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_15': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_16': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_17': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_18': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_19': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_20': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_21': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'point_22': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1

  2%|▏         | 54/2746 [07:12<5:59:17,  8.01s/it]


KeyboardInterrupt: 

In [29]:
arima_predictions

2012-09-11    12.680752
2013-09-11     9.350417
2014-09-11    11.670818
2015-09-11    10.054085
2016-09-10    11.180539
2017-09-10    10.395686
2018-09-10    10.942530
2019-09-10    10.561518
2020-09-09    10.826987
2021-09-09    10.642022
Freq: 365D, Name: predicted_mean, dtype: float64

In [25]:
class ModelTraining():
    """
    Класс для обучения моделей машинного обучения на данных.

    Attributes:
        sql_path(str): Путь к папке с базой данных
    """
    
    def __init__(self, sql_path: str):
        """
        Инициализация класса и подготовка датасета к прогонке через модели.
    
        Attributes:
            sql_path(str): Путь к папке с базой данных
        """

        # Включение игнорирования мешающих предупреждений
        warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)
        # Загрузка базы данных
        try:
            con = sqlite3.connect(sql_path)
            self.data = pd.read_sql("SELECT * FROM Clear_data", con=con)
        except Exception as e:
            print(f"Ошибка в загрузке из базы данных: {e}")
            
        try:
            # Удаление не нужны в обучении колонок
            self.data = self.data.drop(["level_0", "index"], axis=1)
            self.data = self.data.drop(["country", "cluster_2", "point_id"], axis=1)
            # Подготовка колонки со временем, а именно разделение ее на
            # отдельные колонки (date_day, date_month, date_year, date_hour)
            self.data["time"] = pd.to_datetime(self.data["time"])
            self.data['date_day'] = self.data['time'].dt.day
            self.data['date_month'] = self.data['time'].dt.month
            self.data['date_year'] = self.data['time'].dt.year
            self.data['date_hour'] = self.data['time'].dt.hour
            self.data = self.data.drop("time", axis=1)
        except Exception as e:
            print(f"Ошибка в оптимизации данных: {e}")

            
    def fit_model(self, target: str):
        """
        Обучает 3 модели и оценивает их качество
        Args:
            target(str): Название столбца с предказываемыми данными
        """
        
        self.target = target
        X_train, X_test, y_train, y_test = train_test_split(self.data.drop(target, axis=1), self.data[target], test_size=0.33, random_state=42)
        LR = LogisticRegression(max_iter=5000)
        LR.fit(X_train, y_train)
        GBC = GradientBoostingClassifier()
        GBC.fit(X_train, y_train)
        MLPC = MLPClassifier()
        MLPC.fit(X_train, y_train)
        self.models = {"MLPClassifier": MLPC, "GradientBoostingClassifier": GBC, "LogisticRegression": LR}
        for name_model in self.models:
            model = self.models[name_model]
            model_pred = model.predict(X_test)
            model_proba = model.predict_proba(X_test)
            print(f"Модель {name_model}:\n\nТочность: {accuracy_score(y_test, model_pred)}\nПолнота: {recall_score(y_test, model_pred, average="micro")}\nF1-метрика: {f1_score(y_test, model_pred, average="micro")}\nROC-AUC метрика: {roc_auc_score(y_true=y_test, y_score=model_proba[:, 1], average="micro", multi_class="ovr")}\n\n")


    def save_model(self, model_name: str):
        """
        Сохраняет выбранную модель.
        Args:
            model_name(str): Имя выбранной модели
        """

        dump(self.models[model_name], f"{self.target}.joblib")

        
    def __temperature_forecast(self, row):
        """
        С помощью API запроса получается температуру в определенной точке в определенную дату
        Args:
            row(pd.Series): Колонка со всей информацией
        """

        params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "start_date": str(pd.to_datetime(f"{int(row["date_year"])}-{int(row["date_month"])}-{int(row["date_day"])}").strftime("%Y-%m-%d")),
        "end_date": str(pd.to_datetime(f"{int(row["date_year"])}-{int(row["date_month"])}-{int(row["date_day"])}").strftime("%Y-%m-%d")),
        "hourly": ["temperature_2m"]
        }
        
        try:
            openmeteo = openmeteo_requests.Client()
            responses = openmeteo.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)
            response = responses[0]
            current = response.Hourly()
            temperatures = current.Variables(0).ValuesAsNumpy()
            return float(temperatures.mean())
        except Exception as e:
            print(f"Ошибка получения температуры на точке: {(row["latitude"], row["longitude"])}\n{e}")
            return 12

    def predictions(self, target: str, period: int, track: int):
        """
        Получение предсказание модели, на определенный срок вперед.
        Args:
            target(str): Колонка, которую будет предсказывать модель
            period(int): На сколько лет вперед будет предсказание
        """

        
        self.points_predictions_flood = {}
        data = self.data[self.data["track_id"] == track]
        try:
            model = load(f"C:\\Users\\demoexam\\Desktop\\Гурбанов\MachineLearning\\Проект №6 (последний срез)\\{target}.joblib")
        except Exception as e:
            return "Модели, обученной под этот target не было найдено!"
        for i in tqdm.tqdm(range(len(data.index.tolist()))):
            time_column = []
            temperature_column = []
            temperature_range = pd.DataFrame()
            predictions = []
            row = data.iloc[i]
            for k in range(4):
                row["date_year"] -= 1
                time_column.append(str(pd.to_datetime(f"{int(row["date_year"])}-{int(row["date_month"])}-{int(row["date_day"])}").strftime("%Y-%m-%d")))
                temperature_column.append(self.__temperature_forecast(row))
            temperature_range = pd.concat([temperature_range, pd.Series(temperature_column, name="temperature")], axis=1)
            temperature_range = pd.concat([temperature_range, pd.Series(time_column, name="time")], axis=1)
            temperature_range["time"] = pd.to_datetime(temperature_range["time"])
            temperature_range = temperature_range.set_index("time")
            temperature_range = temperature_range.iloc[::-1]
            model = ARIMA(temperature_range, order=(1,1,1))
            model_fit = model.fit()
            arima_predictions = model_fit.forecast(steps=period)
            
            for k, v in enumerate(arima_predictions):
                row = data.iloc[i]
                row["date_year"] += 1
                row["temperature"] = v
                model = load(f"C:\\Users\\demoexam\\Desktop\\Гурбанов\MachineLearning\\Проект №6 (последний срез)\\{target}.joblib")
                predict = model.predict(pd.DataFrame(row).T.drop(target, axis=1))
                predictions.append(int(predict))
            self.points_predictions_flood[f"point_{i}"] = predictions
            
        return self.points_predictions_flood

In [26]:
agent = ModelTraining(r"C:\Users\demoexam\Desktop\Гурбанов\MachineLearning\Проект №6 (последний срез)\clear_data.db")

In [27]:
agent.fit_model("flood")

Модель MLPClassifier:

Точность: 1.0
Полнота: 1.0
F1-метрика: 1.0
ROC-AUC метрика: 1.0


Модель GradientBoostingClassifier:

Точность: 1.0
Полнота: 1.0
F1-метрика: 1.0
ROC-AUC метрика: 1.0


Модель LogisticRegression:

Точность: 1.0
Полнота: 1.0
F1-метрика: 1.0
ROC-AUC метрика: 1.0




In [28]:
agent.save_model("LogisticRegression")

In [29]:
agent.fit_model("fire")

Модель MLPClassifier:

Точность: 1.0
Полнота: 1.0
F1-метрика: 1.0
ROC-AUC метрика: 1.0


Модель GradientBoostingClassifier:

Точность: 1.0
Полнота: 1.0
F1-метрика: 1.0
ROC-AUC метрика: 1.0


Модель LogisticRegression:

Точность: 1.0
Полнота: 1.0
F1-метрика: 1.0
ROC-AUC метрика: 1.0




In [30]:
agent.save_model("LogisticRegression")

In [31]:
agent.predictions("flood", 10, 2)

  0%|          | 0/744 [00:00<?, ?it/s]C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observa

Ошибка получения температуры на точке: (np.float64(51.705434108152986), np.float64(103.42430767603219))
failed to request 'https://archive-api.open-meteo.com/v1/archive': ('Connection aborted.', ConnectionResetError(10054, 'Удаленный хост принудительно разорвал существующее подключение', None, 10054, None))
Ошибка получения температуры на точке: (np.float64(51.705434108152986), np.float64(103.42430767603219))
failed to request 'https://archive-api.open-meteo.com/v1/archive': HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=51.705434108152986&longitude=103.42430767603219&start_date=2018-07-16&end_date=2018-07-16&hourly=temperature_2m&format=flatbuffers (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E552DF4CD0>: Failed to resolve 'archive-api.open-meteo.com' (Name or service not known: archive-api.open-meteo.com using 1 resolver(s))"))
Ошибка получения температуры на точке: (np.flo

C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters f

Ошибка получения температуры на точке: (np.float64(51.705983290448785), np.float64(103.42405227944255))
failed to request 'https://archive-api.open-meteo.com/v1/archive': HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=51.705983290448785&longitude=103.42405227944255&start_date=2021-07-16&end_date=2021-07-16&hourly=temperature_2m&format=flatbuffers (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E552DF4B90>: Failed to resolve 'archive-api.open-meteo.com' (Name or service not known: archive-api.open-meteo.com using 1 resolver(s))"))
Ошибка получения температуры на точке: (np.float64(51.705983290448785), np.float64(103.42405227944255))
failed to request 'https://archive-api.open-meteo.com/v1/archive': HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=51.705983290448785&longitude=103.42405227944255&start_date=2020-07

C:\Users\demoexam\AppData\Local\Temp\ipykernel_6252\4283830431.py:141: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predictions.append(int(predict))
C:\Users\demoexam\AppData\Local\Temp\ipykernel_6252\4283830431.py:141: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predictions.append(int(predict))
C:\Users\demoexam\AppData\Local\Temp\ipykernel_6252\4283830431.py:141: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predictions.append(int(predict))
C:\Users\demoexam\A

...

Ошибка получения температуры на точке: (np.float64(51.73781071789563), np.float64(103.42785950750113))
failed to request 'https://archive-api.open-meteo.com/v1/archive': HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=51.73781071789563&longitude=103.42785950750113&start_date=2021-07-16&end_date=2021-07-16&hourly=temperature_2m&format=flatbuffers (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E553E84410>: Failed to resolve 'archive-api.open-meteo.com' (Name or service not known: archive-api.open-meteo.com using 1 resolver(s))"))
Ошибка получения температуры на точке: (np.float64(51.73781071789563), np.float64(103.42785950750113))
failed to request 'https://archive-api.open-meteo.com/v1/archive': HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=51.73781071789563&longitude=103.42785950750113&start_date=2020-07-16&

C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
C:\Users\demoexam\AppData\Local\Temp\ipykernel_6252\4283830431.py:141: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predictions.append(int(predict))
C:\Users\demoexam\AppData\Local\Temp\ipykernel_6252\4283830431.py:141: DeprecationWarning: Conversion of an array with ndim 

Ошибка получения температуры на точке: (np.float64(51.738099893555045), np.float64(103.42745633795857))
failed to request 'https://archive-api.open-meteo.com/v1/archive': HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=51.738099893555045&longitude=103.42745633795857&start_date=2021-07-16&end_date=2021-07-16&hourly=temperature_2m&format=flatbuffers (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E553E9D590>: Failed to resolve 'archive-api.open-meteo.com' (Name or service not known: archive-api.open-meteo.com using 1 resolver(s))"))
Ошибка получения температуры на точке: (np.float64(51.738099893555045), np.float64(103.42745633795857))
failed to request 'https://archive-api.open-meteo.com/v1/archive': HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=51.738099893555045&longitude=103.42745633795857&start_date=2020-07

C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\demoexam\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
C:\Users\demoexam\AppData\Local\Temp\ipykernel_6252\4283830431.py:141: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predictions.append(int(predict))
C:\Users\demoexam\AppData\Local\Temp\ipykernel_6252\4283830431.py:141: DeprecationWarning: Conversion of an array with ndim 

NameError: name 'points_predictions_flood' is not defined

In [33]:
a = agent.points_predictions_flood

In [36]:
pd.Series(a).T

point_0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
point_1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
point_2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
point_3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
point_4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
                          ...              
point_739    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
point_740    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
point_741    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
point_742    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
point_743    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Length: 744, dtype: object